# Week 6 — adversarial testing (Colab runner)

Windows Application Control blocks torch's DLLs on the dev machine, so the CNN
cannot run locally. Colab has torch preinstalled and no such policy.

This notebook runs **exactly the same scripts** as the local repo — `src/cnn.py`
and `src/adversarial.py`, unmodified. Nothing is reimplemented here, so the
results are comparable to every earlier number in the project.

**Before you start**, build the upload bundle locally:

```powershell
cd C:\dev\phish-detector
Compress-Archive -Path src, data\raw\reference.csv, data\raw\features.csv `
                 -DestinationPath phish-colab.zip -Force
```

That bundle contains no live phishing URLs — `reference.csv` is the 2020
HuggingFace research dataset. `phish_pool.csv` (the live OpenPhish feed) is
deliberately **not** included: it holds currently-active malicious URLs and has
no business being uploaded to a third-party service.

In [ ]:
# 1. Upload phish-colab.zip when the file picker appears.
from google.colab import files

uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 2. Rebuild the repo layout the scripts expect.
#
# Every script resolves paths as Path(__file__).parent.parent, so src/ and
# data/raw/ have to sit under one project root exactly as they do locally.
# Compress-Archive flattens files given by full path, so the CSVs land at the
# top level and get moved back into place here.

!rm -rf /content/phish-detector
!mkdir -p /content/phish-detector/data/raw
!unzip -q -o phish-colab.zip -d /content/phish-detector
!mv -f /content/phish-detector/*.csv /content/phish-detector/data/raw/ 2>/dev/null

%cd /content/phish-detector
!ls -la src data/raw

import torch
print('torch', torch.__version__)

In [ ]:
# 3. Train the CNN. A few minutes on CPU; faster if you set
#    Runtime > Change runtime type > T4 GPU, though the script is CPU-only
#    so the GPU buys nothing here.
#
# Expect: domain-disjoint split, 2587 test URLs, AUC around 0.956.
# If the AUC differs much from 0.956, stop -- something about the data or the
# split did not survive the upload.

!python src/cnn.py

In [ ]:
# 4. The actual week 6 experiment.
#
# Takes the phishing URLs BOTH models currently catch, applies 8 attacker
# edits, and reports how much recall survives each one. Thresholds stay as
# validation chose them -- an attacker does not get to retune your detector.
#
# Prediction on record: camouflage (shared hosting, plain-English paths)
# should beat obfuscation (stripping phish words, flattening paths), because
# obfuscation makes a URL weird and weird is what the models were trained to
# notice. Padding should hurt the CNN and help gradient boosting.

!python src/adversarial.py

In [ ]:
# 5. Download the trained model and its predictions back to the dev machine.
#
# cnn.pt -> models/   (gitignored; week 7's API will load it)
# cnn_test.npz -> reports/   (plot_curves.py and ensemble.py read it)
#
# Both are needed locally: ensemble.py and plot_curves.py run fine without
# torch as long as the CNN's saved predictions are present.

!zip -q -r /content/cnn-outputs.zip models reports/cnn_test.npz
files.download('/content/cnn-outputs.zip')

## Optional: recall against live phishing (needs the pool uploaded)

`live_recall.py` runs the models over `phish_pool.csv` — ~3,500 URLs from the
OpenPhish live feed. The feature-model half already runs locally; only the CNN
needs to come here.

**Read this before running it.** The upload bundle deliberately excludes
`phish_pool.csv` because it holds currently-active malicious URLs. Running this
cell means uploading that file to Colab. The considered view: the pool comes
from a *public* feed and is plain text, so disclosure risk is near zero — but a
Drive-backed runtime holding a file of live phishing URLs can trip Google's
malware scanning, which is an account problem rather than a security one.
Skipping this and running the CNN locally once torch works is the lower-friction
path, and torch has to be fixed for week 7's CLI regardless.

If you want it now: upload `data/raw/phish_pool.csv` via the Files panel, then
run the cell below.

In [ ]:
# Move an uploaded phish_pool.csv into place, then measure live recall.
# Asserts rather than silently reporting a feature-model-only result.

import os

if os.path.exists('/content/phish_pool.csv'):
    !mv -f /content/phish_pool.csv /content/phish-detector/data/raw/

assert os.path.exists('/content/phish-detector/data/raw/phish_pool.csv'), \
    'phish_pool.csv not uploaded -- see the note above'

%cd /content/phish-detector
!python src/live_recall.py

## Back on the dev machine

Unzip `cnn-outputs.zip` into the repo root so `models/cnn.pt` and
`reports/cnn_test.npz` land in place. Then `plot_curves.py` and `ensemble.py`
both work locally again — they only need the CNN's saved predictions, not torch.

Paste the `adversarial.py` output into the chat and we read the results
together. `models/` is gitignored (`*.pt`), so nothing here changes what gets
committed.